# Embeddings

This section generates dense vector representations for all normalized questions using a sentence-transformers bi-encoder, allowing us to capture semantic paraphrases that the lexical MinHash baseline misses. To bypass the impossible computational cost of an exact all-pairs comparison, we build a FAISS HNSW index to efficiently retrieve the top 10-20 nearest neighbors for each question based on cosine similarity.

In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
import numpy as np
import faiss
from torch.utils.data import DataLoader
import time
from scipy.spatial.distance import cdist
import os

In [6]:
print(os.getcwd())

df = pd.read_parquet('../data/processed/questions.parquet')
texts = df['text_norm'].fillna("").tolist()
qids = df['qid'].tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True)

np.save('embeddings.npy', embeddings)

c:\Users\Alberto\Desktop\Computational Tools\quora-question-pairs\notebooks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2099 [00:00<?, ?it/s]

In [7]:
d = embeddings.shape[1]
index_hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
index_hnsw.add(embeddings)

k = 20
threshold = 0.80

scores, indices = index_hnsw.search(embeddings, k)

candidates = []
for i in range(len(qids)):
    for j in range(1, k):
        score = float(scores[i][j])
        if score >= threshold:
            candidates.append({
                'qid_a': qids[i],
                'qid_b': qids[indices[i][j]],
                'score': score,
                'source': 'emb'
            })

df_candidates = pd.DataFrame(candidates)
df_candidates.to_parquet('candidates_emb.parquet', index=False)

In [8]:
sample_size = min(1000, len(embeddings))
sample_queries = embeddings[:sample_size]

index_exact = faiss.IndexFlatIP(d)
index_exact.add(embeddings)

exact_scores, exact_indices = index_exact.search(sample_queries, k)
hnsw_scores, hnsw_indices = index_hnsw.search(sample_queries, k)

hits = 0
for i in range(sample_size):
    exact_set = set(exact_indices[i])
    hnsw_set = set(hnsw_indices[i])
    hits += len(exact_set.intersection(hnsw_set))

recall = hits / (sample_size * k)
print(f"HNSW Recall: {recall:.4f}")

HNSW Recall: 0.9615
